# Refresh Download of Reviews

In [1]:
import requests
import pandas as pd


# Get top 100 games by player count from SteamSpy
response = requests.get('https://steamspy.com/api.php?request=top100in2weeks')
top_games = pd.DataFrame(response.json()).T #shortand for JSON transpose rows to cols
top_games = top_games.head(100) #get first 100 rows
top_games['appid'] = top_games['appid'].astype(int) #convert appid to integer
top_games[['appid', 'name']].head() #get first 5 rows of appid and name

,appid,name
730,730,Counter-Strike: Global Offensive
1172470,1172470,Apex Legends
578080,578080,PUBG: BATTLEGROUNDS
1623730,1623730,Palworld
440,440,Team Fortress 2


In [2]:
!pip install -Uqq steam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.1/644.1 kB 9.8 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [3]:
import os
from steam.webapi import WebAPI
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
STEAM_API_KEY = user_secrets.get_secret("STEAM_API_KEY")

api = WebAPI(key=STEAM_API_KEY)

# Example: Get reviews for a single app
def get_reviews(appid, num_reviews=100, cursor='*'):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    params = {
        'json': 1,
        'num_per_page': num_reviews,
        'cursor': cursor,
        'filter': 'recent',
        'language': 'english'
    }
    r = requests.get(url, params=params)
    if r.status_code == 200:
        data = r.json()
        if 'reviews' in data:
            return { 'cursor': data['cursor'], 'reviews': data['reviews'] }
    return []

In [4]:
# Preview of CS review JSON data
cs_reviews = get_reviews(730, num_reviews=1)
cs_reviews['reviews'][0]

{'recommendationid': '199301103',
 'author': {'steamid': '76561199856697403',
  'num_games_owned': 0,
  'num_reviews': 1,
  'playtime_forever': 919,
  'playtime_last_two_weeks': 168,
  'playtime_at_review': 919,
  'last_played': 1751994371},
 'language': 'english',
 'review': 'This game measurably shortens your life span and takes away your soul.',
 'timestamp_created': 1751994396,
 'timestamp_updated': 1751994396,
 'voted_up': False,
 'votes_up': 0,
 'votes_funny': 0,
 'weighted_vote_score': 0.5,
 'comment_count': 0,
 'steam_purchase': True,
 'received_for_free': False,
 'written_during_early_access': False,
 'primarily_steam_deck': False}

In [5]:
from tqdm import tqdm

all_reviews = []
for _, row in tqdm(top_games.iterrows(), total=top_games.shape[0]):
    appid = row['appid']
    name = row['name']
    response = get_reviews(appid, num_reviews=100)
    for review in response['reviews']:
        all_reviews.append({
            'appid': appid,
            'name': name,
            'review': review['review'],
            'timestamp_created': review['timestamp_created'],
            'voted_up': review['voted_up'],
            'votes_up': review['votes_up'],
            'votes_funny': review['votes_funny'],
            'weighted_vote_score': review['weighted_vote_score'],
        })

reviews_df = pd.DataFrame(all_reviews)
reviews_df.head()

100%|██████████| 100/100 [00:35<00:00,  2.78it/s]


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,This game measurably shortens your life span a...,1751994396,False,0,0,0.5
1,730,Counter-Strike: Global Offensive,Decent,1751994216,True,0,0,0.5
2,730,Counter-Strike: Global Offensive,so bad,1751994192,True,0,0,0.5
3,730,Counter-Strike: Global Offensive,https://steamcommunity.com/profiles/7656119798...,1751994048,True,0,0,0.5
4,730,Counter-Strike: Global Offensive,zıbestinzıworldolum,1751993846,True,0,0,0.5


In [6]:
reviews_df.to_csv('reviews.csv', index=False)

# Start Here for analysis

In [7]:
import pandas as pd

df = pd.read_csv('reviews.csv')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9801 entries, 0 to 9800
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   appid                9801 non-null   int64  
 1   name                 9801 non-null   object 
 2   review               9750 non-null   object 
 3   timestamp_created    9801 non-null   int64  
 4   voted_up             9801 non-null   bool   
 5   votes_up             9801 non-null   int64  
 6   votes_funny          9801 non-null   int64  
 7   weighted_vote_score  9801 non-null   float64
dtypes: bool(1), float64(1), int64(4), object(2)
memory usage: 545.7+ KB


appid: The unique steam id for each game   
name: The unique game name  
review: The corpus of all text in the review  
timestamp_created: Unix time (epoch) in UTC (POSIX TIME) seconds since 01/01/1970  
`voted_up`: The reviewer's thumb up (True) or thumb down (False) Score `(our target)`  
votes_up: Number of people who upvoted the review  
votes_funny: Number of people who thought the vote was funny  
weighted_vote_score: Steam's helpfullness score - between 0 to 1 - where low scores are likely spam  


In [9]:
df.weighted_vote_score.describe()

count    9801.000000
mean        0.502606
std         0.027544
min         0.203082
25%         0.500000
50%         0.500000
75%         0.500000
max         0.918151
Name: weighted_vote_score, dtype: float64

Looking at the weighted score - we can see the first 75% essentially stay at or below the 50% probability of spam. Why don't we only grab reviews that are substantial by filtering the df to where the score is above 0.50.

In [10]:
df_filtered = df[df.weighted_vote_score > 0.6]
len(df_filtered)

92

Ok, that took us from almost 10k results down to 1571.. we probably should request around 100 with this criteria before filtering - but let's observed some of the comments in the filtered. 

We checked at >= 0.5 and that results in 8771 remaining.. so quite a few right at 0.5 are removed as expected.

Also at >0.6 results in only 104

In [11]:
df_filtered.head()

,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
126,1172470,Apex Legends,GAYPEX LEGENDS,1731639079,False,9,2,0.619687
141,1172470,Apex Legends,Game is actually broken now in season 22. This...,1723184775,False,171,15,0.804729
630,1063730,New World: Aeternum,Terrible community full of toxicity. God forb...,1751594893,False,26,5,0.725189
1119,1599340,Lost Ark,It was super fun at the start playing and expl...,1750002721,False,56,0,0.801719
1143,1599340,Lost Ark,"At launch, Lost Ark was a thrilling experience...",1747522319,False,80,0,0.837621


In [12]:
# Pre-filter Reviews
quality_reviews = []
for _, row in tqdm(top_games.iterrows(), total=top_games.shape[0]):
    n_quality = 0
    appid = row['appid']
    name = row['name']
    cursor = '*' # Initial cursor
    
    while n_quality < 90:
        reviews_data = get_reviews(appid, num_reviews=100, cursor=cursor)
        reviews = reviews_data['reviews']
        cursor = reviews_data.get('cursor') # Update's cursor value

        if not reviews:
            break # early escape if no more reviews to fetch
        
        for review in reviews:
            if float(review['weighted_vote_score']) >= 0.51:
                n_quality += 1
                all_reviews.append({
                    'appid': appid,
                    'name': name,
                    'review': review['review'],
                    'timestamp_created': review['timestamp_created'],
                    'voted_up': review['voted_up'],
                    'votes_up': review['votes_up'],
                    'votes_funny': review['votes_funny'],
                    'weighted_vote_score': review['weighted_vote_score'],
                })
            

quality_reviews_df = pd.DataFrame(all_reviews)
quality_reviews_df.head()

100%|██████████| 100/100 [05:23<00:00,  3.24s/it]


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,This game measurably shortens your life span a...,1751994396,False,0,0,0.5
1,730,Counter-Strike: Global Offensive,Decent,1751994216,True,0,0,0.5
2,730,Counter-Strike: Global Offensive,so bad,1751994192,True,0,0,0.5
3,730,Counter-Strike: Global Offensive,https://steamcommunity.com/profiles/7656119798...,1751994048,True,0,0,0.5
4,730,Counter-Strike: Global Offensive,zıbestinzıworldolum,1751993846,True,0,0,0.5


In [13]:
# check in on counts and if filtering now
print("Total reviews:", len(quality_reviews_df))

has_word = quality_reviews_df['review'].str.contains(r'\b\w+\b')
is_long = quality_reviews_df.review.str.len().ge(10)
english_only = quality_reviews_df['review'].str.fullmatch(r"[A-Za-z0-9\s.,!?\"'’\-():;]+", na=False)

filtered_df = quality_reviews_df[has_word & is_long & english_only]
print("Filtered reviews (≥10 chars and has english words):", len(filtered_df))

Total reviews: 18848
Filtered reviews (≥10 chars and has english words): 12487


In [14]:
# Get the remaining games that still have at least 100 reviews
game_counts = filtered_df.name.value_counts()
min_reviews = game_counts[game_counts >= 100].index

min_df = filtered_df[filtered_df.name.isin(min_reviews)]
min_df.name.value_counts()

name
Z1 Battle Royale                         160
Half-Life 2: Lost Coast                  153
7 Days to Die                            152
Monster Hunter Wilds                     150
Grand Theft Auto IV: Complete Edition    150
                                        ... 
Black Myth: Wukong                       113
Wallpaper Engine                         113
eFootball                                113
Grand Theft Auto V Legacy                109
VRChat                                   103
Name: count, Length: 93, dtype: int64

In [15]:
min_df.to_csv("quality_reviews.csv")

Ok, now we have a reasonable amount of data with some context around the game, the review, and the voted up tag.  

We're going to generate a corpus of input and then split the full data into 3 sets- train, validate and test - in 80:10:10 batches

# Start Here for Clean Analysis


In [16]:
# Start here for clean
import pandas as pd
min_df = pd.read_csv("quality_reviews.csv")

In [17]:
# Clean the punctuation
import re

def cleaned(text):
    return re.sub(r'\W+', '_', text).lower()

In [18]:
# Add input Field
df = min_df.copy().reset_index(drop=True)
df['input'] = 'TEXT1: ' + df.name.apply(cleaned) + '; TEXT2: ' + df.review

# Convert the target to boolean ints
# Our target is currently saved as boolean true false - so let's convert to int
df['voted_up'] = df['voted_up'].astype(int)

display(df['input'].head())
display(df.voted_up.head())

0               TEXT1: apex_legends; TEXT2: Good Shit!
1    TEXT1: apex_legends; TEXT2: Never in my life w...
2    TEXT1: apex_legends; TEXT2: Best gunfeel acros...
3    TEXT1: apex_legends; TEXT2: This game is super...
4    TEXT1: apex_legends; TEXT2: Supposed anti-chea...
Name: input, dtype: object

0    1
1    0
2    1
3    1
4    0
Name: voted_up, dtype: int64

In [19]:
# Now let's get experience with datasets ( required for hugging face transformers )
from datasets import Dataset,DatasetDict

# Select the columns we want to keep for the dataset/prediction purposes
columns = ['input', 'voted_up']
df = df[columns].copy()

ds = Dataset.from_pandas(df)

In [20]:
ds

Dataset({
    features: ['input', 'voted_up'],
    num_rows: 12047
})

In [21]:
# We're going to create todenizers using deberta
model_name = 'microsoft/deberta-v3-small'

!pip install -Uq transformers
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_name)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 88.5 MB/s eta 0:00:00:00:01:01


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


The warning is only an issue because we're using noisy or multilingual data - the reviews, even when marked english, have all kinds of randomness.

Our options are to ignore (if we're ok with slighly less flexible outcomes) - instead of <unk> the characters would be byte level split.  
Or we can use something like..  
> from transformers import T5Tokenizer  
> tokenizer = T5Tokenizer.from_pretrained("t5-base", use_fast=False)  

In [22]:
# Test out the tokenizer with some basic text
tokz.tokenize("TEXT1: An Hello, I'm new at this and learning is fun!")

['▁TEXT',
 '1',
 ':',
 '▁An',
 '▁Hello',
 ',',
 '▁I',
 "'",
 'm',
 '▁new',
 '▁at',
 '▁this',
 '▁and',
 '▁learning',
 '▁is',
 '▁fun',
 '!']

In [23]:
# Vs some other text in the head of an earlier preview
tokz.tokenize("Tässä pelissä on intensiivistä väkivaltaa")

['▁T',
 'ä',
 's',
 's',
 'ä',
 '▁pe',
 'liss',
 'ä',
 '▁on',
 '▁in',
 't',
 'ensi',
 'ivist',
 'ä',
 '▁vä',
 'k',
 'ival',
 'ta',
 'a']

The tokenizer works fine on normal text but not fine on other text

In [24]:
# simple function to tokenize our
def tok_func(x): return tokz(x["input"])

In [25]:
# Parallel for every row in ds with map
tok_ds = ds.map(tok_func, batched=True)

Map:   0%|          | 0/12047 [00:00<?, ? examples/s]

In [26]:
row = tok_ds[0]
row['input'], row['input_ids']

('TEXT1: apex_legends; TEXT2: Good Shit!',
 [1,
  54453,
  435,
  294,
  24376,
  616,
  88591,
  268,
  346,
  54453,
  445,
  294,
  1798,
  55819,
  300,
  2])

These ids are a list of vocab in the tokenizer with a unique int for every string

In [27]:
# See the int above
tokz.vocab['▁An']

816

In [28]:
# Transformers needs a column called labels
tok_ds = tok_ds.rename_columns({'voted_up':'labels'})

In [29]:
tok_ds

Dataset({
    features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 12047
})

`tok_ds` is now ready for splitting

In [30]:
# Step 1: Train/Test Split (e.g., 80% train, 20% temp)
train_test = tok_ds.train_test_split(test_size=0.2, seed=42)

# Step 2: Split test portion into validation and test (e.g., 50/50 of the 20%)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

train_ds = train_test['train']
test_ds = val_test['train']
eval_ds = val_test['test']

dds = DatasetDict({
    'train': train_ds,
    'test': test_ds,
    'eval': eval_ds
})

In [31]:
dds

DatasetDict({
    train: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9637
    })
    test: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1205
    })
    eval: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1205
    })
})

In [32]:
# To train our model in transformers we need to more modules
from transformers import TrainingArguments, Trainer

2025-07-08 17:51:46.533415: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751997106.727162      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751997106.785594      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [33]:
bs = 128 # batch size
epochs = 4
lr = 8e-5 # small learning rate, halve the size if too far

In [34]:
import transformers
from transformers import TrainingArguments
print(transformers.__version__)

4.53.1


In [39]:
args = TrainingArguments(
    'outputs',
    learning_rate=lr,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=2,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to='none'
)

In [36]:
# Define how to compute the metric or target by SGD
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

In [40]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Slow things down but reduce memory usage
model.gradient_checkpointing_enable()

trainer = Trainer(
    model, 
    args, 
    train_dataset=dds['train'], 
    eval_dataset=dds['test'],
    tokenizer=tokz, 
    compute_metrics=compute_metrics
)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_35/165106228.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Not Enough Memory to continue on Kaggle
For whatever reason - running this fills up the available GPU memory very quickly.  

Probably there is some type of smaller process or batch process that would help process, then clear, then next batch the request.

In [41]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
# Get raw logits
raw_preds = trainer.predict(eval_ds)

# Convert logits to class predictions (0 or 1)
preds = np.argmax(raw_preds.predictions, axis=1)

# If needed, also get ground truth labels
true_labels = raw_preds.label_ids

# Optionally inspect
print(preds[:10])